# Explore the eval — what passes, what fails, and why

This notebook is your window into how the agent **scored** on the FinanceBench starter set.
The agent answers a question, a hardened LLM **judge** grades the answer against the known-correct
("gold") answer, and an independent **panel of 3 blind graders** double-checks the judge.

**Setup:** pick the project's `.venv` as the kernel (kernel picker → `.venv\\Scripts\\python.exe`).
All logic lives in `src/sec_filings/`; this notebook only displays it. **Edit the strings and re-run cells.**

Sections:
1. The scorecard — the headline numbers
2. Visualize — pass/fail by company
3. The full table — every question at a glance (fails in red)
4. Inspect one question in depth — gold vs agent vs judge vs panel
5. Test the agent yourself — ask anything live

In [ ]:
import mistune
from IPython.display import HTML, display

from sec_filings.inspection import eval_records
from sec_filings.config import settings


def show_md(text, height=420):
    """Render markdown (e.g. an agent answer) as a scrollable HTML box."""
    html = mistune.html(text or "")
    display(HTML(
        f'<div style="max-height:{height}px;overflow:auto;padding:10px 14px;'
        f'border:1px solid #ddd;border-radius:8px;line-height:1.55">{html}</div>'
    ))


# One row per question: question + gold + agent answer + judge verdict + panel verdict.
records = eval_records()
by_qid = {r["qid"]: r for r in records}
print(f"Loaded {len(records)} graded questions from results/eval_run.json")
print("judge model:", settings.judge_model, " | agent model:", settings.agent_model)

## 1. The scorecard

`grade_correct` is the **hardened judge's** verdict; `panel_correct` is the **independent 3-grader
panel's** majority verdict. When they agree, you can trust the number.

In [ ]:
n = len(records)
judge_pass = sum(1 for r in records if r["grade_correct"])
panel_pass = sum(1 for r in records if r["panel_correct"])
agree = sum(1 for r in records
            if r["panel_correct"] is not None and bool(r["grade_correct"]) == bool(r["panel_correct"]))

print(f"Agent accuracy (judge):     {judge_pass}/{n} = {judge_pass / n:.0%}")
print(f"Independent panel:          {panel_pass}/{n} = {panel_pass / n:.0%}")
print(f"Judge vs panel agreement:   {agree}/{n}   <- how much you can trust the score")
print()
print("By question type:")
for t in ("domain-relevant", "novel-generated", "metrics-generated"):
    sub = [r for r in records if r["type"] == t]
    if sub:
        ok = sum(1 for r in sub if r["grade_correct"])
        print(f"  {t:<18} {ok}/{len(sub)}")

## 2. Visualize — pass / fail by company

Green = the judge graded the agent's answer correct; red = incorrect. Hover for counts.

In [ ]:
import plotly.graph_objects as go

tickers = sorted({r["ticker"] for r in records})
passes = [sum(1 for r in records if r["ticker"] == t and r["grade_correct"]) for t in tickers]
fails = [sum(1 for r in records if r["ticker"] == t and not r["grade_correct"]) for t in tickers]

fig = go.Figure([
    go.Bar(name="pass", x=tickers, y=passes, marker_color="#16a34a"),
    go.Bar(name="fail", x=tickers, y=fails, marker_color="#dc2626"),
])
fig.update_layout(barmode="stack", title="Eval results by company (judge verdict)",
                  height=380, yaxis_title="questions", legend_title_text="")
fig.show()

## 3. The full table — every question at a glance

Fails are red. `panel` shows the blind panel's vote (e.g. `0/3` = all three graders said incorrect,
`3/3` = unanimous correct). Copy any `id` into Section 4 to dig in.

In [ ]:
def scorecard_table(records):
    head = ("<tr style='text-align:left;border-bottom:2px solid #333'>"
            "<th style='padding:4px 10px'>co</th><th style='padding:4px 10px'>id</th>"
            "<th style='padding:4px 10px'>type</th><th style='padding:4px 10px'>verdict</th>"
            "<th style='padding:4px 10px'>panel</th><th style='padding:4px 10px'>judge reason</th></tr>")
    rows = []
    for r in records:
        ok = r["grade_correct"]
        color = "#16a34a" if ok else "#dc2626"
        mark = "PASS" if ok else "FAIL"
        panel = r.get("panel_votes") or "-"
        reason = (r.get("grade_reason") or "")[:95]
        rows.append(
            "<tr style='border-bottom:1px solid #eee'>"
            f"<td style='padding:4px 10px'>{r['ticker']}</td>"
            f"<td style='padding:4px 10px;font-family:monospace'>{r['qid'][-5:]}</td>"
            f"<td style='padding:4px 10px'>{r['type']}</td>"
            f"<td style='padding:4px 10px;color:{color};font-weight:600'>{mark}</td>"
            f"<td style='padding:4px 10px'>{panel}</td>"
            f"<td style='padding:4px 10px;color:#555'>{reason}</td></tr>"
        )
    display(HTML(f"<table style='border-collapse:collapse;font-size:0.85rem'>{head}{''.join(rows)}</table>"))


scorecard_table(records)

## 4. Inspect one question in depth

The whole story for a single question: the gold answer, the agent's full answer (rendered),
the judge's reasoning + issues, and what the 3 blind graders said. **Edit `QID` and re-run** —
it accepts a full id or just the last 5 digits (e.g. `"01328"`).

It defaults to **PEP 01328**, the restructuring question the old judge nearly passed by mistake
(agent said "$0", gold is $411M). Other interesting ones: `00222` (AMD quick ratio), `00476`
(AXP hedge), `03620` (a hard multi-statement calc the agent nails).

In [ ]:
QID = "01328"   # <-- edit me: any id from the table above

key = QID if QID in by_qid else next((k for k in by_qid if k.endswith(QID)), None)
if key is None:
    raise KeyError(f"No question matching {QID!r}. Pick an id from Section 3.")
r = by_qid[key]

verdict = "PASS" if r["grade_correct"] else "FAIL"
print(f"{r['ticker']}  {r['qid']}  [{r['type']}]")
print(f"JUDGE: {verdict}     PANEL: {r.get('panel_votes')} correct")
print("=" * 92)
print("QUESTION:\n ", r["question"])
print("\nGOLD ANSWER:\n ", r["gold_answer"])
print("\nJUDGE REASON:\n ", r["grade_reason"])
if r.get("grade_issues"):
    print("\nJUDGE ISSUES:")
    for i in r["grade_issues"]:
        print("  -", i)
if r.get("grader_notes"):
    print("\nINDEPENDENT PANEL (3 blind graders):")
    for note in r["grader_notes"]:
        print("  -", note)
print("\nAGENT ANSWER (rendered below):")
show_md(r["agent_answer"])

## 5. Test the agent yourself

Ask anything — not just eval questions. Indexed filings: **MSFT 2023, AMD/AXP/BA/PEP 2022, AMCR 2023**
(it'll tell you if you ask about a filing it doesn't have). A compound question makes several tool
calls and can take 20–40s; the `live` hook prints each step so you can watch it work.

In [ ]:
from sec_filings.agent.loop import run_agent

QUESTION = "What were PepsiCo's restructuring costs in FY2022?"   # <-- edit me


def live(step):
    if step.type == "tool_call":
        print(f"  -> {step.tool_name}({step.tool_input})", flush=True)
    elif step.type == "tool_result":
        print(f"  <- {step.tool_name} returned", flush=True)
    elif step.type == "answer":
        print("  done.\n", flush=True)


print(f"Q: {QUESTION}\n")
trace = run_agent(QUESTION, on_event=live)
show_md(trace.final_answer)

---
**Want to see retrieval and chunk boundaries?** That lives in `notebooks/01_explore.ipynb`
(raw hybrid retrieval) and the Streamlit explorer: `uv run streamlit run app/explorer.py`.
This notebook stays focused on the eval.